<a href="https://colab.research.google.com/github/Siam007-ayuno/Python_Practice/blob/main/Wikipedia_pageviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import requests
import pandas as pd

url = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/Bangladesh/daily/2026090100/2026112600"

headers = {"User-Agent": "SAIF-ML-Lesson/1.0 (contact: saifayuno@gmail.com)"}
data = requests.get(url, headers=headers).json()

df = pd.DataFrame(data["items"])
df["date"] = pd.to_datetime(df["timestamp"], format="%Y%m%d%H")
df = df[["date", "views"]]
print(df.head())

        date  views
0 2026-09-01   7165
1 2026-09-02   6949
2 2026-09-03   7180
3 2026-09-04   6570
4 2026-09-05   6516


In [18]:
df["rolling_7"] = df["views"].rolling(window=7).mean().shift(1)
df["yesterday"] = df["views"].shift(1)
df["last_week"] = df["views"].shift(7)
df["weekday"] = df["date"].dt.weekday
df = df.dropna().reset_index(drop=True)

In [19]:
from sklearn.linear_model import LinearRegression

X = df[["yesterday", "last_week", "weekday", "rolling_7"]]
y = df["views"]
X_train, y_train = X[:-14], y[:-14]
X_test, y_test = X[-14:], y[-14:]

model = LinearRegression()
model.fit(X_train, y_train)
print("Train score:", model.score(X_train, y_train))
print("Test score:", model.score(X_test, y_test))

Train score: 0.22887280023843504
Test score: -0.17313284936273465


In [20]:
from sklearn.metrics import mean_absolute_error
pred = model.predict(X_test)
print("Average error:", round(mean_absolute_error(y_test, pred)), "views")

Average error: 377 views
